IMPORTS

In [1]:
import pandas as pd
import numpy as np
import string
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

DATA LOADING

In [2]:
df = pd.read_csv(r'ArticlesApril2017.csv')

EXTRACTING SENTENCES

In [3]:
sentences = [
    sentence 
    for sentence in df.headline if sentence != "Unknown"
    ]

CLEANING SENTENCES

In [4]:
sentences = [
    "".join(
        ch 
        for ch in sentence if ch not in string.punctuation
    ) 
    for sentence in sentences
]

SETTING UP TOKENIZER

In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
total_words = len(tokenizer.word_index) + 1

PREFIX N-GRAM GENERATION

In [6]:
sequences = []
for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    
    for i in range(1,len(token_list)):
        sequences.append(token_list[:i+1])

PADDING SEQUENCES

In [7]:
longest = max(len(seq) for seq in sequences)
sequences = pad_sequences(
    sequences,
    maxlen= longest,
    padding="pre"
)

ONE HOT REPRESENTATION

In [8]:
X = sequences[:,:-1]
y = to_categorical(
    sequences[:,-1],
    num_classes = total_words
    )

CREATING MODEL

In [ ]:
model = Sequential([
                    Embedding( 
                                total_words,
                                10,
                                input_length = longest - 1
                            ),
                    LSTM(100),
                    Dense(
                            total_words,    
                            activation = "softmax"
                        )
])


c:\Users\exerc\Documents\NLP\NLP\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
model.compile(
            loss='categorical_crossentropy',
            optimizer='adam'
    )

In [11]:
model.fit(
    X,
    y,
    epochs = 20, 
    verbose = 0
)

In [12]:
def generate_text(seed,next_words):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed])[0]

        pad = pad_sequences(
                            [seq],
                            maxlen = longest - 1,
                            padding = "pre"
                            )
        
        pred = np.argmax(
            model.predict(pad),
            axis = 1
        )

        for word, index in tokenizer.word_index.items():
            if index == pred:
                seed += " " + word
                break
    return seed

In [13]:
print(generate_text("united states", 10))
print(generate_text("president trump", 10))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
united states york a focus on the end of a front editor
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
president trump is a border of a next door of john edgar
